In [995]:
# 0. import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates
import datetime
import json
from pathlib import Path

In [996]:
# read cleaned data set
folder = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPT_MRT/rawdata/_combined")

df_4hour_step = pd.read_csv(folder / 'hourly_step_counts.csv')
df_yesterday_step = pd.read_csv(folder / 'yesterday_step_counts.csv')
df_prior2hours_step = pd.read_csv(folder / 'prior_2hours_step_counts.csv')
df_otherPA = pd.read_csv(folder / 'recorded_physical_activity.csv')

df_weekly = pd.read_csv(folder / 'df_weekly_filled.csv')
df_daily = pd.read_csv(folder / 'df_daily_filled.csv')

df_daily_pageview = pd.read_csv(folder / 'df_daily_pageview.csv')
df_hourly_pageview = pd.read_csv(folder / 'hourly_pageview.csv')


df_gif = pd.read_csv(folder / 'df_gif_all.csv')
df_salience = pd.read_csv(folder / 'df_salience_all.csv')
df_planning = pd.read_csv(folder / 'df_end_all.csv')

df_notwearing = pd.read_csv(folder / 'missing_days.csv')
df_wearing_morning = pd.read_csv(folder / 'wear_day.csv')

In [997]:
print(df_weekly)

     ParticipantIdentifier        date observed_date  week_present  \
0                      118  2025-09-21    2025-09-22             1   
1                      118  2025-09-28           NaN             0   
2                      118  2025-10-05    2025-10-06             1   
3                      118  2025-10-12    2025-10-12             1   
4                      118  2025-10-19    2025-10-20             1   
..                     ...         ...           ...           ...   
127                    225  2026-02-08    2026-02-08             1   
128                    225  2026-02-15    2026-02-15             1   
129                    225  2026-02-22    2026-02-22             1   
130                    225  2026-03-01    2026-03-01             1   
131                    225  2026-03-08    2026-03-08             1   

     AffectiveValuation  CAE-1  CAE-10  CAE-11  CAE-12  CAE-2  CAE-3  CAE-4  \
0                   5.0    6.0     5.0     5.0     5.0    6.0    5.0    6.0   
1

In [998]:
# create a data frame to contain outcome and predictors for hourly step counts
df_4hour_step['Date'] = pd.to_datetime(df_4hour_step['Date'])
df_yesterday_step['Date'] = pd.to_datetime(df_yesterday_step['Date'])
df_prior2hours_step['Date'] = pd.to_datetime(df_prior2hours_step['Date'])
df_otherPA['Date'] = pd.to_datetime(df_otherPA['Date'])

df_weekly['Date'] = pd.to_datetime(df_weekly['date'])
df_daily['Date'] = pd.to_datetime(df_daily['date'])

df_daily_pageview['Date'] = pd.to_datetime(df_daily_pageview['Date'])
df_hourly_pageview['Date'] = pd.to_datetime(df_hourly_pageview['Date'])

df_gif['Date'] = pd.to_datetime(df_gif['Date'])
df_salience['Date'] = pd.to_datetime(df_salience['Date'])
df_planning['Date'] = pd.to_datetime(df_planning['date'])

df_notwearing['Date'] = pd.to_datetime(df_notwearing['Date'])
df_wearing_morning['Date'] = pd.to_datetime(df_wearing_morning['Date'])


In [999]:
print(f"Before merging with other data: {len(df_4hour_step)} rows")

Before merging with other data: 2024 rows


In [1000]:
# Start with hourly step data
df_merged = df_4hour_step.copy()
print(df_merged[150:200])

# 1) Merge daily data - map to all hours of the same date
# daily_cols = [col for col in df_daily.columns if col not in ['participantidentifier', 'date', 'observed_date']]
# df_merged = df_merged.merge(
#     df_daily[['participantidentifier', 'date'] + daily_cols],
#     on=['participantidentifier', 'date'],
#     how='left'
# )

# 2) Add yesterday's daily present, affective reflection, and anticipated affect
# yesterday_cols = ['daily_present', 'affective_reflection', 'anticipated_affect']
# df_yesterday = df_daily[['participantidentifier', 'date'] + yesterday_cols].copy()

# Shift date forward by 1 day (so yesterday's values map to today)
# df_yesterday['date'] = df_yesterday['date'] + pd.Timedelta(days=1)

# Rename columns to indicate they're from yesterday
# df_yesterday.columns = ['participantidentifier', 'date'] + [f'yesterday_{col}' for col in yesterday_cols]

# Merge yesterday's values
# df_merged = df_merged.merge(
#     df_yesterday,
#     on=['participantidentifier', 'date'],
#     how='left'
# )

# print(df_merged[df_merged['participantidentifier'] == 31])

     ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
150                    118 2025-11-21             0  2025-11-21 08:30:00   
151                    118 2025-11-21             1  2025-11-21 13:30:00   
152                    118 2025-11-22             0  2025-11-22 08:30:00   
153                    118 2025-11-22             1  2025-11-22 13:30:00   
154                    118 2025-11-23             0  2025-11-23 08:30:00   
155                    118 2025-11-23             1  2025-11-23 13:30:00   
156                    118 2025-11-24             0  2025-11-24 08:30:00   
157                    118 2025-11-24             1  2025-11-24 13:30:00   
158                    118 2025-11-25             0  2025-11-25 08:30:00   
159                    118 2025-11-25             1  2025-11-25 13:30:00   
160                    118 2025-11-26             0  2025-11-26 08:30:00   
161                    118 2025-11-26             1  2025-11-26 13:30:00   
162         

In [1001]:
# merge with yesterday step counts
df_merged = df_merged.merge(
    df_yesterday_step,
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)

print(df_merged.head(50))

    ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                     118 2025-09-07             0  2025-09-07 08:30:00   
1                     118 2025-09-07             1  2025-09-07 13:30:00   
2                     118 2025-09-08             0  2025-09-08 08:30:00   
3                     118 2025-09-08             1  2025-09-08 13:30:00   
4                     118 2025-09-09             0  2025-09-09 08:30:00   
5                     118 2025-09-09             1  2025-09-09 13:30:00   
6                     118 2025-09-10             0  2025-09-10 08:30:00   
7                     118 2025-09-10             1  2025-09-10 13:30:00   
8                     118 2025-09-11             0  2025-09-11 08:30:00   
9                     118 2025-09-11             1  2025-09-11 13:30:00   
10                    118 2025-09-12             0  2025-09-12 08:30:00   
11                    118 2025-09-12             1  2025-09-12 13:30:00   
12                    118

In [1002]:
# merge with prior 2hours step counts
df_merged = df_merged.merge(
    df_prior2hours_step,
    on=['ParticipantIdentifier', 'Date', 'DecisionTime'],
    how='left'
)

print(df_merged)

      ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                       118 2025-09-07             0  2025-09-07 08:30:00   
1                       118 2025-09-07             1  2025-09-07 13:30:00   
2                       118 2025-09-08             0  2025-09-08 08:30:00   
3                       118 2025-09-08             1  2025-09-08 13:30:00   
4                       118 2025-09-09             0  2025-09-09 08:30:00   
...                     ...        ...           ...                  ...   
2019                    225 2026-03-07             1  2026-03-07 12:30:00   
2020                    225 2026-03-08             0  2026-03-08 07:30:00   
2021                    225 2026-03-08             1  2026-03-08 12:30:00   
2022                    225 2026-03-09             0  2026-03-09 07:30:00   
2023                    225 2026-03-09             1  2026-03-09 12:30:00   

      StepCount_x  CheckStatus_x  EMA_StepCount  YesterdayStepCount  \
0   

In [1003]:
# merge with other PA data
df_merged = df_merged.merge(
    df_otherPA,
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)


In [1004]:
# merge with not wearing data
df_merged = df_merged.merge(
    df_notwearing,
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)


In [1005]:
# merge with wearing data
df_merged = df_merged.merge(
    df_wearing_morning,
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)


In [1006]:
# merge with daily pageview counts
df_merged = df_merged.merge(
    df_daily_pageview,
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)

print(df_merged)

      ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                       118 2025-09-07             0  2025-09-07 08:30:00   
1                       118 2025-09-07             1  2025-09-07 13:30:00   
2                       118 2025-09-08             0  2025-09-08 08:30:00   
3                       118 2025-09-08             1  2025-09-08 13:30:00   
4                       118 2025-09-09             0  2025-09-09 08:30:00   
...                     ...        ...           ...                  ...   
2019                    225 2026-03-07             1  2026-03-07 12:30:00   
2020                    225 2026-03-08             0  2026-03-08 07:30:00   
2021                    225 2026-03-08             1  2026-03-08 12:30:00   
2022                    225 2026-03-09             0  2026-03-09 07:30:00   
2023                    225 2026-03-09             1  2026-03-09 12:30:00   

      StepCount_x  CheckStatus_x  EMA_StepCount  YesterdayStepCount  \
0   

In [1007]:
# merge with hourly pageview counts
df_merged = df_merged.merge(
    df_hourly_pageview,
    on=['ParticipantIdentifier', 'Date', 'DecisionTime'],
    how='left'
)


In [1008]:
# 2) Merge weekly data - use only one date per week to avoid duplicates
df_weekly['week'] = pd.to_datetime(df_weekly['date']).dt.isocalendar().week
df_weekly['year'] = pd.to_datetime(df_weekly['date']).dt.isocalendar().year

# Keep only one row per participant-week-year (the first date of each week)
df_weekly_unique = df_weekly.groupby(
    ['ParticipantIdentifier', 'week', 'year'], as_index=False
).first()

df_merged['week'] = df_merged['Date'].dt.isocalendar().week
df_merged['year'] = df_merged['Date'].dt.isocalendar().year

# print(df_merged[df_merged['ParticipantIdentifier'] == "118"])

weekly_cols = [col for col in df_weekly_unique.columns 
               if col not in ['ParticipantIdentifier', 'date', 'week', 'year', 'observed_date']]

df_merged = df_merged.merge(
    df_weekly_unique[['ParticipantIdentifier', 'week', 'year'] + weekly_cols],
    on=['ParticipantIdentifier', 'week', 'year'],
    how='left'
)
print(f"After weekly merge: {len(df_merged)} rows")

print(df_weekly.week.unique())
print(df_merged.week.unique())
# print(df_merged.iloc[5100:5150, :26])

After weekly merge: 2024 rows
<IntegerArray>
[38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52,  1,  2,  3,  4,
  5,  6,  7,  8,  9, 10]
Length: 25, dtype: UInt32
<IntegerArray>
[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52,  1,  2,
  3,  4,  5,  6,  7,  8,  9, 10, 11]
Length: 28, dtype: UInt32


In [1009]:


# remove duplicate Date columns
df_merged = df_merged.drop(columns=['Date_y'])

# change column names Date_x and Date_y to Date
df_merged = df_merged.rename(columns={'Date_x': 'Date'})

print(df_merged[df_merged['ParticipantIdentifier'] == 141]['week'].unique())



<IntegerArray>
[42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 1, 2, 3]
Length: 14, dtype: UInt32


In [1010]:
# Define columns to lag
lag_columns = weekly_cols

# Create lagged columns
for col in lag_columns:
    if col in df_weekly_unique.columns:
        df_weekly_unique[f'{col}_lastweek'] = df_weekly_unique.groupby('ParticipantIdentifier')[col].shift(1)


# Now merge the lagged columns into df_merged
lastweek_cols = [col for col in df_weekly_unique.columns if 'lastweek' in col]

df_merged = df_merged.merge(
    df_weekly_unique[['ParticipantIdentifier', 'week', 'year'] + lastweek_cols],
    on=['ParticipantIdentifier', 'week', 'year'],
    how='left',
    suffixes=('', '_dup')
)

print(df_merged)
# print(df_merged[df_merged['participantidentifier'] == 13].iloc[:50, :26])


      ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                       118 2025-09-07             0  2025-09-07 08:30:00   
1                       118 2025-09-07             1  2025-09-07 13:30:00   
2                       118 2025-09-08             0  2025-09-08 08:30:00   
3                       118 2025-09-08             1  2025-09-08 13:30:00   
4                       118 2025-09-09             0  2025-09-09 08:30:00   
...                     ...        ...           ...                  ...   
2019                    225 2026-03-07             1  2026-03-07 12:30:00   
2020                    225 2026-03-08             0  2026-03-08 07:30:00   
2021                    225 2026-03-08             1  2026-03-08 12:30:00   
2022                    225 2026-03-09             0  2026-03-09 07:30:00   
2023                    225 2026-03-09             1  2026-03-09 12:30:00   

      StepCount_x  CheckStatus_x  EMA_StepCount  YesterdayStepCount  \
0   

In [1011]:
# merge with daily survey data
# df_merged = df_merged.merge(
#     df_daily,
#     on=['ParticipantIdentifier', 'Date'],
#     how='left'
# )

# add lag columns
lag_columns = ['daily_present', 'affective_reflection', 'anticipated_affect']

for col in lag_columns:
    df_daily[f'{col}_yesterday'] = df_daily.groupby('ParticipantIdentifier')[col].shift(1)

df_merged = df_merged.merge(
    df_daily[['ParticipantIdentifier', 'Date'] + lag_columns + [f'{col}_yesterday' for col in lag_columns]],
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)

print(df_merged)

      ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                       118 2025-09-07             0  2025-09-07 08:30:00   
1                       118 2025-09-07             1  2025-09-07 13:30:00   
2                       118 2025-09-08             0  2025-09-08 08:30:00   
3                       118 2025-09-08             1  2025-09-08 13:30:00   
4                       118 2025-09-09             0  2025-09-09 08:30:00   
...                     ...        ...           ...                  ...   
2019                    225 2026-03-07             1  2026-03-07 12:30:00   
2020                    225 2026-03-08             0  2026-03-08 07:30:00   
2021                    225 2026-03-08             1  2026-03-08 12:30:00   
2022                    225 2026-03-09             0  2026-03-09 07:30:00   
2023                    225 2026-03-09             1  2026-03-09 12:30:00   

      StepCount_x  CheckStatus_x  EMA_StepCount  YesterdayStepCount  \
0   

In [1012]:
# merging with action delivery data

# 1) Merge planning prompts- map to all hours of the same date
planning_cols = [col for col in df_planning.columns if col not in ['ParticipantIdentifier', 'Date', 'time']]
# df_merged["ParticipantIdentifier"] = pd.to_numeric(df_merged["ParticipantIdentifier"], errors="coerce").astype("Int64")
df_planning["ParticipantIdentifier"] = pd.to_numeric(df_planning["ParticipantIdentifier"], errors="coerce").astype("Int64")
print(df_planning.ParticipantIdentifier.unique())
df_merged = df_merged.merge(
    df_planning[['ParticipantIdentifier', 'Date'] + planning_cols],
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)
print(df_merged.shape)

# # 2) add yesterday's end of day survey
df_yesterday = df_planning[['ParticipantIdentifier', 'Date'] + planning_cols].copy()
df_yesterday['Date'] = df_yesterday['Date'] + pd.Timedelta(days=1)
df_yesterday.columns = ['ParticipantIdentifier', 'Date'] + [f'yesterday_{col}' for col in planning_cols]

df_merged = df_merged.merge(df_yesterday, on=['ParticipantIdentifier', 'Date'], how='left')

# print(df_merged[df_merged['participantidentifier'] == 31])

<IntegerArray>
[118, 141, 143, 151, 160, 170, 184, 188, 195, 204, 225]
Length: 11, dtype: Int64
(2024, 68)


In [1013]:
print(df_merged)

      ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                       118 2025-09-07             0  2025-09-07 08:30:00   
1                       118 2025-09-07             1  2025-09-07 13:30:00   
2                       118 2025-09-08             0  2025-09-08 08:30:00   
3                       118 2025-09-08             1  2025-09-08 13:30:00   
4                       118 2025-09-09             0  2025-09-09 08:30:00   
...                     ...        ...           ...                  ...   
2019                    225 2026-03-07             1  2026-03-07 12:30:00   
2020                    225 2026-03-08             0  2026-03-08 07:30:00   
2021                    225 2026-03-08             1  2026-03-08 12:30:00   
2022                    225 2026-03-09             0  2026-03-09 07:30:00   
2023                    225 2026-03-09             1  2026-03-09 12:30:00   

      StepCount_x  CheckStatus_x  EMA_StepCount  YesterdayStepCount  \
0   

In [1014]:
# Merge with walking data
# print(df_walking.head())

# 1. Build an hourly view of the walking data
# df_walking['datetime'] = pd.to_datetime(
#     df_walking['date'].dt.strftime('%Y-%m-%d') + ' ' + df_walking['time']
# )

# walking_hourly = (
#     df_walking
#     .sort_values('datetime')
#     .groupby(['participantidentifier', 'date', 'hour'], as_index=False)
#     .agg({'walking_suggestion': 'max', 'open': 'max'})  # max==1 if any event in that hour
# )

# print(walking_hourly.head())

# 2. Merge into the existing hourly panel
df_merged = df_merged.merge(
    df_gif,
    on=['ParticipantIdentifier', 'Date', 'DecisionTime'],
    how='left'
)

# 3. Fill missing hours with 0 (no walking suggestion delivered in that hour)
# df_merged['walking_suggestion'] = df_merged['walking_suggestion'].fillna(0).astype(int)
# df_merged['open'] = df_merged['open'].fillna(0).astype(int)

print(df_merged.head(60))

    ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                     118 2025-09-07             0  2025-09-07 08:30:00   
1                     118 2025-09-07             1  2025-09-07 13:30:00   
2                     118 2025-09-08             0  2025-09-08 08:30:00   
3                     118 2025-09-08             1  2025-09-08 13:30:00   
4                     118 2025-09-09             0  2025-09-09 08:30:00   
5                     118 2025-09-09             1  2025-09-09 13:30:00   
6                     118 2025-09-10             0  2025-09-10 08:30:00   
7                     118 2025-09-10             1  2025-09-10 13:30:00   
8                     118 2025-09-11             0  2025-09-11 08:30:00   
9                     118 2025-09-11             1  2025-09-11 13:30:00   
10                    118 2025-09-12             0  2025-09-12 08:30:00   
11                    118 2025-09-12             1  2025-09-12 13:30:00   
12                    118

In [1015]:
# Merge with salience data
print(df_salience.head())

# 1. Build an hourly view of the salience data
salience_cols = [col for col in df_salience.columns if col not in ['ParticipantIdentifier', 'Date', 'time']]
df_merged = df_merged.merge(
    df_salience[['ParticipantIdentifier', 'Date'] + salience_cols],
    on=['ParticipantIdentifier', 'Date'],
    how='left'
)

# 2. add yesterday's salience data
df_yesterday = df_salience[['ParticipantIdentifier', 'Date'] + salience_cols].copy()
df_yesterday['Date'] = df_yesterday['Date'] + pd.Timedelta(days=1)
df_yesterday.columns = ['ParticipantIdentifier', 'Date'] + [f'yesterday_{col}' for col in salience_cols]

df_merged = df_merged.merge(df_yesterday, on=['ParticipantIdentifier', 'Date'], how='left')

print(df_merged[df_merged['ParticipantIdentifier'] == 118])


   ParticipantIdentifier       Date      Time  SalienceMessage  Interacted  \
0                    118 2025-09-14  11:45:00                0           0   
1                    118 2025-09-15  11:45:11                1           1   
2                    118 2025-09-16  11:45:00                0           0   
3                    118 2025-09-17  11:45:11                1           1   
4                    118 2025-09-18  11:45:07                1           1   

   Interacted_7d  
0            NaN  
1            NaN  
2            NaN  
3            NaN  
4            NaN  
     ParticipantIdentifier       Date  DecisionTime        DateTimeStart  \
0                      118 2025-09-07             0  2025-09-07 08:30:00   
1                      118 2025-09-07             1  2025-09-07 13:30:00   
2                      118 2025-09-08             0  2025-09-08 08:30:00   
3                      118 2025-09-08             1  2025-09-08 13:30:00   
4                      118 2025-09-09

In [1016]:
# check variables
print(df_merged.columns)

Index(['ParticipantIdentifier', 'Date', 'DecisionTime', 'DateTimeStart',
       'StepCount_x', 'CheckStatus_x', 'EMA_StepCount', 'YesterdayStepCount',
       'StepCount_y', 'CheckStatus_y', 'RecordedPhysicalActivity',
       'Previous7DaysRPA', 'DayWearing', 'ValidHours', 'past7days_daywearing',
       'nextday_wearing', 'restday_valid_minutes', 'morning_wearing',
       'restday_wearing', 'DailyPageviewCount', 'YesterdayPageviewCount',
       'Past7DaysPageviewEMA', 'TomorrowPageviewCount', 'DatetimeStart',
       'HourlyPageviewCount', 'week', 'year', 'week_present',
       'AffectiveValuation', 'CAE-1', 'CAE-10', 'CAE-11', 'CAE-12', 'CAE-2',
       'CAE-3', 'CAE-4', 'CAE-5', 'CAE-6', 'CAE-7', 'CAE-8', 'CAE-9',
       'Exp-tool-1', 'Exp-tool-2', 'week_present_lastweek',
       'AffectiveValuation_lastweek', 'CAE-1_lastweek', 'CAE-10_lastweek',
       'CAE-11_lastweek', 'CAE-12_lastweek', 'CAE-2_lastweek',
       'CAE-3_lastweek', 'CAE-4_lastweek', 'CAE-5_lastweek', 'CAE-6_lastweek',


In [1017]:
# drop columns date_x, date_y, Time_x, Time_y
df_merged = df_merged.drop(columns=['date', 'Time_x', 'Time_y'])

# modify the column names
df_merged = df_merged.rename(columns={'Interacted_x': 'Interacted_walk', 'Interacted_y': 'Interacted_salience', 
                                     'Interacted_7d_x': 'Interacted_7d_walk', 'Interacted_7d_y': 'Interacted_7d_salience',
                                     'StepCount_x': '4hour_step', 'StepCount_y': 'prior2hour_step'})


In [1018]:
# check the number of dates per participant
print(df_merged.groupby('ParticipantIdentifier')['Date'].nunique())

ParticipantIdentifier
118    92
141    92
143    92
151    92
160    92
170    92
184    92
188    92
195    92
204    92
225    92
Name: Date, dtype: int64


In [1019]:
# Remove the first 7 calendar days of data for each participant
df_merged = df_merged.copy()
df_merged["Date"] = pd.to_datetime(df_merged["Date"], errors="coerce").dt.normalize()

min_date = df_merged.groupby("ParticipantIdentifier")["Date"].transform("min")
cutoff = min_date + pd.Timedelta(days=7)

df_merged = df_merged[df_merged["Date"] >= cutoff].reset_index(drop=True)
print(df_merged.loc[df_merged['ParticipantIdentifier'] == 118, 'week'].iloc[0:50])

0     37
1     37
2     38
3     38
4     38
5     38
6     38
7     38
8     38
9     38
10    38
11    38
12    38
13    38
14    38
15    38
16    39
17    39
18    39
19    39
20    39
21    39
22    39
23    39
24    39
25    39
26    39
27    39
28    39
29    39
30    40
31    40
32    40
33    40
34    40
35    40
36    40
37    40
38    40
39    40
40    40
41    40
42    40
43    40
44    41
45    41
46    41
47    41
48    41
49    41
Name: week, dtype: UInt32


In [1020]:
# start from 0 for each participant
# Calendar days since first study date (normalize so two same-day decisions share one `day`)
_d = pd.to_datetime(df_merged['Date']).dt.normalize()
df_merged['day'] = (
    df_merged.assign(_dn=_d)
    .sort_values(['ParticipantIdentifier', 'Date'])
    .groupby('ParticipantIdentifier')['_dn']
    .transform(lambda d: (d - d.min()).dt.days)
)
# print(df_merged.day)

# Study week index (aligned with `day`): days 0-6 → week 0, 7-13 → week 1, …
# With ~2 decision rows per calendar day, expect ~14 consecutive rows per `week` (7 days × 2).
df_merged = df_merged.sort_values(['ParticipantIdentifier', 'Date'])
df_merged['week'] = (df_merged['day'] // 7).astype(int)
_ex = (
    df_merged[df_merged['ParticipantIdentifier'] == 141][['Date', 'day', 'week']]
    .head(50)
    .reset_index(drop=True)
)
print(_ex)

# 1) Day-of-week number: Monday=1 … Sunday=7
df_merged['dow'] = df_merged['Date'].dt.dayofweek + 1

# 2) Weekend flag (1 = Saturday/Sunday, else 0)
df_merged['is_weekend'] = (df_merged['dow'] >= 6).astype(int)


df_merged = df_merged.sort_values(
    ['ParticipantIdentifier', 'Date', 'DateTimeStart']
)

# df_merged['hour_in_day'] = (
#     df_merged.groupby(['participantidentifier', 'date'])
#              .cumcount() + 1
# )

# creat CAE_avg that averages CAE_1 to CAE_12 
# and CAE_avg_lastweek that averages CAE_1_lastweek to CAE_12_lastweek of the last week
df_merged['CAE_avg'] = df_merged[['CAE-1', 'CAE-2', 'CAE-3', 'CAE-4', 'CAE-5', 'CAE-6', 'CAE-7', 'CAE-8', 
                                 'CAE-9', 'CAE-10', 'CAE-11', 'CAE-12']].mean(axis=1)
df_merged['CAE_avg_lastweek'] = df_merged[['CAE-1_lastweek', 
                                          'CAE-2_lastweek', 'CAE-3_lastweek', 'CAE-4_lastweek', 
                                          'CAE-5_lastweek', 'CAE-6_lastweek', 'CAE-7_lastweek', 
                                          'CAE-8_lastweek', 'CAE-9_lastweek', 'CAE-10_lastweek', 
                                          'CAE-11_lastweek', 'CAE-12_lastweek']].mean(axis=1)
df_merged['CAE_short_avg'] = df_merged[['CAE-1', 'CAE-3', 'CAE-7']].mean(axis=1)

df_merged['Perceived_utility'] = df_merged[['Exp-tool-1', 'Exp-tool-2']].mean(axis=1)
df_merged['Perceived_utility_lastweek'] = df_merged[['Exp-tool-1_lastweek', 'Exp-tool-2_lastweek']].mean(axis=1)

# create a new column called Exponentially weighted average of 4 hour step counts over the past 7 days
window = 7  # seven mornings (or afternoons)

df_merged = df_merged.sort_values(["ParticipantIdentifier", "Date", "DecisionTime"])

# recent burden: row-wise mean of walk / salience / planning interaction flags, then exponentially
# weighted *within each participant* (rows must be time-ordered). span=14 rows ≈ 7 calendar days
# if there are 2 decision rows per day; use span=7 if you mean 7 decision rows instead.
_burden_cols = ["WalkingSuggestion",  "yesterday_SalienceMessage", "yesterday_planning_prompt"]
df_merged["recent_burden"] = (
    df_merged.assign(_instant_burden=df_merged[_burden_cols].mean(axis=1))
    .groupby("ParticipantIdentifier")["_instant_burden"]
    .transform(lambda s: s.ewm(span=14, adjust=False, min_periods=1).mean())
)




# ewm_step = (
#     df_merged
#     .groupby(['ParticipantIdentifier', 'DecisionTime'])['step_count']
#     .transform(lambda s: s.ewm(span=window, min_periods=1).mean())
# )
# df_merged['7day_step_count_avg'] = ewm_step

# # create a new column called Standard deviation of 4 hour step counts over the past 7 days
# df_merged['7day_step_count_std'] = (
#     df_merged
#     .groupby(['ParticipantIdentifier', 'DecisionTime'])['step_count']
#     .transform(lambda s: s.rolling(window=window, min_periods=2).std())
# )

# week_step_count_avg = (
#     df_merged
#     .groupby(['ParticipantIdentifier', 'week'])['step_count']
#     .transform(lambda s: s.ewm(span=len(s), min_periods=1).mean())
# )

# df_merged['week_step_count_avg'] = week_step_count_avg

# week_walking_suggestion = (
#     df_merged
#     .groupby(['ParticipantIdentifier', 'week', 'DecisionTime'])['walking_suggestion']
#     .transform(lambda s: s.ewm(span=len(s), min_periods=1).mean())
# )

# df_merged['week_walking_suggestion'] = week_walking_suggestion

# week_view_status = (
#     df_merged
#     .groupby(['ParticipantIdentifier', 'week'])['view_status']
#     .transform(lambda s: s.ewm(span=len(s), min_periods=1).mean())
# )

# df_merged['week_view_status'] = week_view_status

# df_merged['view_status_lastdecision'] = (
#     df_merged
#     .groupby('participantidentifier')['view_status']
#     .shift(1)
# )

#

         Date  day  week
0  2025-10-20    0     0
1  2025-10-20    0     0
2  2025-10-21    1     0
3  2025-10-21    1     0
4  2025-10-22    2     0
5  2025-10-22    2     0
6  2025-10-23    3     0
7  2025-10-23    3     0
8  2025-10-24    4     0
9  2025-10-24    4     0
10 2025-10-25    5     0
11 2025-10-25    5     0
12 2025-10-26    6     0
13 2025-10-26    6     0
14 2025-10-27    7     1
15 2025-10-27    7     1
16 2025-10-28    8     1
17 2025-10-28    8     1
18 2025-10-29    9     1
19 2025-10-29    9     1
20 2025-10-30   10     1
21 2025-10-30   10     1
22 2025-10-31   11     1
23 2025-10-31   11     1
24 2025-11-01   12     1
25 2025-11-01   12     1
26 2025-11-02   13     1
27 2025-11-02   13     1
28 2025-11-03   14     2
29 2025-11-03   14     2
30 2025-11-04   15     2
31 2025-11-04   15     2
32 2025-11-05   16     2
33 2025-11-05   16     2
34 2025-11-06   17     2
35 2025-11-06   17     2
36 2025-11-07   18     2
37 2025-11-07   18     2
38 2025-11-08   19     2


In [1021]:
print(df_merged.loc[df_merged['ParticipantIdentifier'] == 141, 'week'])
# print(df_merged.columns)

170     0
171     0
172     0
173     0
174     0
       ..
335    11
336    11
337    11
338    12
339    12
Name: week, Length: 170, dtype: int64


In [1022]:
# save the merged data
df_merged.to_csv(folder / 'df_merged.csv', index=False)